# Two incidences fill a rectangle
### A construction page for reciprocal floor sums

For positive **coprime** integers $a,b>1$,

\[
\boxed{\sum_{x=1}^{b-1}\left\lfloor\frac{ax}{b}\right\rfloor
+\sum_{y=1}^{a-1}\left\lfloor\frac{by}{a}\right\rfloor
=(a-1)(b-1).}
\]

For $a=11,b=7$, the two sums are $30+30=60$.
Here **area** means discrete area: one unit cell per integer pair in the stated domain.
The displayed squares represent these cells; their centers determine incidence.
This is not a claim about the integrals of the floor functions over real intervals.

We will define two incidences, inspect their counts, and bring them together. The
mathematical argument below establishes the general identity; the code checks exact
finite cases. The notation beside each figure is an exposition of the visible Python
construction, not a new executable text grammar.

Use the **Kaleion** kernel and **Restart Kernel and Run All Cells**. Installation is
the same as in [the first notebook](01_discovery_workbench.ipynb).

In [ ]:
from pathlib import Path
from math import gcd
import json

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display

from kaleion import Collection, F, Motion, Workspace, choose, param, vector
from kaleion.viewers.plotly import animation_figure

pio.renderers.default = "plotly_mimetype+notebook"
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists() and (p / "src/kaleion").is_dir())
OUTPUT = ROOT / "build" / "notebooks" / "floor-sum"
OUTPUT.mkdir(parents=True, exist_ok=True)
PARAMETERS = {"a": 11, "b": 7}  # Try {"a": 12, "b": 8} after the default example.
assert all(isinstance(v, int) and not isinstance(v, bool) and v > 1
           for v in PARAMETERS.values())
a, b = param("a"), param("b")

## 1 · The earlier counts are exactly floor quotients

In the first notebook, for each $0\le i<b$,

\[
c_i=\#\{j:0\le j<a,\; ai+bj\ge ab\}.
\]

Set $y=a-j$. As $j$ runs from $0$ to $a-1$, $y$ runs through $a,a-1,\ldots,1$, and

\[
ai+bj\ge ab \iff b(a-j)\le ai \iff y\le ai/b.
\]

Since $0\le ai/b<a$, there are exactly $\lfloor ai/b\rfloor$ such positive integers
$y$. This includes $i=0$, whose count is zero. **Coprimality is not needed here.**
The coordinate change $(j,-i)\mapsto(i,a-j)$ makes the incidence look like the
familiar columns under a floor function.

In [ ]:
region = Collection.grid(b, a, values=a * F.i + b * F.j).arrange(F.j, -F.i)
original_incidence = region.where(F.value >= a * b)
original_counts = original_incidence.count(by=F.i)
actual = original_counts.evaluate(**PARAMETERS).values.tolist()
expected = [PARAMETERS["a"] * i // PARAMETERS["b"] for i in range(PARAMETERS["b"])]
assert actual == expected
print("Original row counts:", actual)
print("Exact floor quotients:", expected)

## 2 · Define one universe and two incidences

\[
\begin{aligned}
D&=\{1,\ldots,b-1\}\times\{1,\ldots,a-1\},\\
P&=\{(x,y)\in D:by\le ax\},\\
Q&=\{(x,y)\in D:ax\le by\}.
\end{aligned}
\]

For a fixed $x$, $P$ contains $\lfloor ax/b\rfloor$ points. For a fixed $y$, $Q$
contains $\lfloor by/a\rfloor$ points. Both lenses are applied to the **same**
arrangement, so intersection and union compare the same occurrences.

In Python, `u` and `v` are exact integer fields for mathematical $x$ and $y$;
`F.x` and `F.y` denote floating placement coordinates and are not used to decide
these inequalities. Every occurrence is assigned the value 1 to represent unit weight.

In [ ]:
rectangle = (Collection.grid(b - 1, a - 1, values=1)
             .annotate(u=F.i + 1, v=F.j + 1)
             .arrange(F.u, F.v))
P = rectangle.where(b * F.v <= a * F.u)
Q = rectangle.where(a * F.u <= b * F.v)
columns = P.count(by=F.u)
rows = Q.count(by=F.v)

ROOTS = {
    "D": rectangle, "P": P, "Q": Q,
    "union": P | Q, "overlap": P & Q,
    "columns": columns, "rows": rows,
    "sum_columns": columns.sum(), "sum_rows": rows.sum(),
    "area": rectangle.count(),
}
workspace = Workspace(ROOTS, PARAMETERS)
assert not workspace.state.errors, workspace.state.errors
state = workspace.state
print("Counts by x:", state.results["columns"].values.tolist())
print("Counts by y:", state.results["rows"].values.tolist())

In [ ]:
# These checks concern the selected finite case. The proof follows below.
def inspect_case(state):
    if state.errors:
        raise ValueError(dict(state.errors))
    results = state.results
    av, bv = state.parameters["a"], state.parameters["b"]
    left = int(results["sum_columns"].values[0])
    right = int(results["sum_rows"].values[0])
    area = int(results["area"].values[0])
    overlap = results["overlap"].cardinality
    assert results["D"].ids == results["P"].source.ids == results["Q"].source.ids
    assert results["union"].mask.all()
    assert results["union"].cardinality == area == (av - 1) * (bv - 1)
    assert left == results["P"].cardinality
    assert right == results["Q"].cardinality
    assert results["columns"].values.tolist() == [av * x // bv for x in range(1, bv)]
    assert results["rows"].values.tolist() == [bv * y // av for y in range(1, av)]
    assert left + right - overlap == area
    assert overlap == gcd(av, bv) - 1
    return {"a": av, "b": bv, "sum_P": left, "sum_Q": right,
            "overlap": overlap, "area": area, "covers": True,
            "disjoint": overlap == 0, "case_verified": True}

report = inspect_case(state)
print(f"Count(P) + Count(Q) − Count(P ∩ Q) = {report['sum_P']} + {report['sum_Q']} − {report['overlap']} = {report['area']}")

## 3 · Read the incidences beside the figure

Use **P**, **Q**, **Both**, or **Overlap** to inspect the two sets and their shared
domain. The line is a geometric guide; membership is calculated by exact integer
inequalities on the cell centers. Display gaps between squares are only styling.

The following cell contains the plotting helper and is initially collapsed. It reads
captured results; it does not construct relations or recompute their counts.

In [ ]:
COLORS = {"empty": "#253348", "P": "#47bfa7", "Q": "#eeb65d", "both": "#c392ee"}

def incidence_page(state):
    r = state.results
    if state.errors:
        raise ValueError(dict(state.errors))
    av, bv = state.parameters["a"], state.parameters["b"]
    nx, ny = bv - 1, av - 1
    p, q = r["P"].mask, r["Q"].mask
    count_p = int(r["sum_columns"].values[0])
    count_q = int(r["sum_rows"].values[0])
    overlap = r["overlap"].cardinality
    area = int(r["area"].values[0])
    codes = p.astype(int) + 2 * q.astype(int)

    def grid(values):
        return np.asarray(values).reshape(nx, ny).T.tolist()

    palette = [[0, COLORS["empty"]], [1/6, COLORS["empty"]],
               [1/6, COLORS["P"]], [0.5, COLORS["P"]],
               [0.5, COLORS["Q"]], [5/6, COLORS["Q"]],
               [5/6, COLORS["both"]], [1, COLORS["both"]]]
    memberships = grid(["P and Q" if v == 3 else "P" if v == 1 else "Q" for v in codes])
    fig = go.Figure(go.Heatmap(
        x=list(range(1, bv)), y=list(range(1, av)), z=grid(codes),
        customdata=memberships, zmin=0, zmax=3, colorscale=palette,
        showscale=False, xgap=2, ygap=2,
        hovertemplate="x = %{x}, y = %{y}<br>In the full construction: %{customdata}<extra></extra>",
    ))
    # Clip the guide to the rectangle of cell centers; no float decides membership.
    lo = max(1, bv / av)
    hi = min(bv - 1, bv * (av - 1) / av)
    fig.add_trace(go.Scatter(x=[lo, hi], y=[av * lo / bv, av * hi / bv],
                            mode="lines", line=dict(color="#e6edf7", width=1, dash="dot"),
                            hoverinfo="skip", showlegend=False))
    conclusion = (f"Disjoint cover<br>{count_p} + {count_q} = {area}" if overlap == 0 else
                  f"Cover with overlap<br>{count_p} + {count_q} − {overlap} = {area}")
    text = (f"<b>Given</b>  a = {av}, b = {bv}<br>"
            f"D = {{1,…,{bv-1}}} × {{1,…,{av-1}}}<br><br>"
            "<b>Mark</b>  P : by ≤ ax<br><b>Mark</b>  Q : ax ≤ by<br><br>"
            f"<b>Count P by x</b><br>Σ floor(ax/b) = {count_p}<br>"
            "x = 1,…,b−1<br><br>"
            f"<b>Count Q by y</b><br>Σ floor(by/a) = {count_q}<br>"
            "y = 1,…,a−1<br><br>"
            f"<b>{conclusion}</b>")
    stages = {"P": p.astype(int), "Q": 2 * q.astype(int),
              "Both": codes, "Overlap": 3 * (p & q).astype(int)}
    fig.update_layout(
        template="plotly_dark", height=600, paper_bgcolor="#101827", plot_bgcolor="#101827",
        title=dict(text="Two incidences · one declared universe", x=0.04),
        margin=dict(l=28, r=28, t=112, b=85),
        xaxis=dict(domain=[0.48, 1], title="x", range=[0.5, bv-0.5], dtick=1, constrain="domain"),
        yaxis=dict(title="y", range=[0.5, av-0.5], dtick=1, scaleanchor="x", scaleratio=1, constrain="domain"),
        annotations=[dict(x=0, y=1, xref="paper", yref="paper", xanchor="left", yanchor="top",
                          text=text, showarrow=False, align="left", font=dict(size=14)),
                     dict(x=0.5, y=-0.14, xref="paper", yref="paper", showarrow=False,
                          text="Teal: P · Gold: Q · Purple: shared in Both view", font=dict(size=12))],
        updatemenus=[dict(type="buttons", direction="right", x=0.48, xanchor="left", y=1.02, yanchor="bottom", active=2,
                          bgcolor="#293449", showactive=False,
                          buttons=[dict(label=name, method="restyle", args=[{"z": [grid(values)]}, [0]])
                                   for name, values in stages.items()])],
    )
    return fig

In [ ]:
partition_plot = incidence_page(state)
partition_plot.show()

## 4 · Bring the two parts together

For the coprime case, place $Q$ to the right of $P$, then move it into its original
slots. Every occurrence keeps its identity. At the endpoint, each slot has exactly
one item. The animation illustrates the disjoint cover established by the incidences.

During the motion, overlapping display markers do not mean an additional mathematical
intersection. Membership and counts come from the captured endpoint construction.
If the selected parameters are not coprime, this packing illustration is omitted;
the overlap is inspected explicitly in the next section.

In [ ]:
packing_plot = None
if report["disjoint"]:
    separated = rectangle.move(vector(choose(a * F.u <= b * F.v, b, 0), 0))
    packing_workspace = Workspace({"tiles": separated}, PARAMETERS)
    packed = packing_workspace.set("tiles", rectangle, motion=Motion())
    unpacked = packing_workspace.undo()
    times = np.linspace(0, 1, 41)
    frames = [step.frame("tiles", float(t)) for step in (packed, unpacked) for t in times]
    labels = [f"{verb} · {t:.0%}" for verb in ("Fill", "Separate") for t in times]
    packing_plot = animation_figure(frames, labels=labels, show_values=False,
                                    title=f"{report['sum_P']} + {report['sum_Q']} unit cells fill {report['area']} slots",
                                    duration=50)
    packing_plot.update_xaxes(constrain="domain")
    packing_plot.update_yaxes(constrain="domain")
    for i, slider_step in enumerate(packing_plot.layout.sliders[0].steps):
        slider_step.label = labels[i]
    colors_by_id = {oid: COLORS["P"] if selected else COLORS["Q"]
                    for oid, selected in zip(state.results["D"].ids, state.results["P"].mask)}
    # Style tracks by captured identity, never by interpolated position or label.
    for plotted, sampled in zip([packing_plot, *packing_plot.frames], [frames[0], *frames]):
        plotted.data[0].marker.update(symbol="square", size=19,
                                      color=[colors_by_id[oid] for oid in sampled.after_ids])
    np.testing.assert_allclose(packed.frame("tiles", 1).positions,
                               state.results["D"].positions, rtol=0, atol=1e-12)
    packing_plot.show()
else:
    display(Markdown("The selected incidences overlap. Use the inclusion–exclusion view below."))

## 5 · Why this is a proof for every coprime pair

**Count each incidence.** At fixed $x$, the integers in $P$ are
$1\le y\le\lfloor ax/b\rfloor$. Therefore
$|P|=\sum_{x=1}^{b-1}\lfloor ax/b\rfloor$.
At fixed $y$, the integers in $Q$ are
$1\le x\le\lfloor by/a\rfloor$, so
$|Q|=\sum_{y=1}^{a-1}\lfloor by/a\rfloor$.

**Cover the universe.** Every pair of integers $ax,by$ satisfies $by\le ax$ or
$ax\le by$. Thus $P\cup Q=D$.

**Show disjointness.** A point in both sets would satisfy $ax=by$. If
$\gcd(a,b)=1$, divisibility gives $b\mid x$, impossible for $1\le x<b$.
Consequently $P\cap Q=\varnothing$.

**Count the partition.** The Cartesian domain has $(b-1)(a-1)$ occurrences. Hence

\[
\sum_{x=1}^{b-1}\left\lfloor\frac{ax}{b}\right\rfloor
+\sum_{y=1}^{a-1}\left\lfloor\frac{by}{a}\right\rfloor
=|P|+|Q|=|D|=(a-1)(b-1).
\]

This is the general argument. The assertions and animation above instantiate it for
chosen parameters; they are not an automated proof certificate.

## 6 · Relax the assumption and inspect what changes

Let $d=\gcd(a,b)$. Points in $P\cap Q$ satisfy $ax=by$, hence

\[
(x,y)=\left(k\frac{b}{d},k\frac{a}{d}\right),\qquad k=1,\ldots,d-1.
\]

There are $d-1$ such points. The two **non-strict** incidences still cover $D$, but
each shared point contributes to both sums. Inclusion–exclusion now gives

\[
\boxed{\sum_{x=1}^{b-1}\left\lfloor\frac{ax}{b}\right\rfloor
+\sum_{y=1}^{a-1}\left\lfloor\frac{by}{a}\right\rfloor
=(a-1)(b-1)+\gcd(a,b)-1.}
\]

For $(a,b)=(12,8)$, the sums give $40+40=80$. The rectangle contains $77$ cells
and there are three shared centers: $(2,3),(4,6),(6,9)$. The extra three are
double counts, not additional area. Click **Overlap** to isolate them.

In [ ]:
noncoprime_workspace = Workspace(ROOTS, {"a": 12, "b": 8})
noncoprime_report = inspect_case(noncoprime_workspace.state)
overlap_snapshot = noncoprime_workspace.state.results["overlap"]
print("Shared coordinates:", overlap_snapshot.source.positions[overlap_snapshot.mask].astype(int).tolist())
print(noncoprime_report)
noncoprime_plot = incidence_page(noncoprime_workspace.state)
noncoprime_plot.show()

## 7 · Keep the investigation and its derivation

The union, intersection, grouped counts, and their sums are explicit construction
roots. Saving this workspace retains the finite states and their provenance. The
written proof supplies the general reasoning; the capture records what this
particular evaluation established.

In [ ]:
workspace.capture(
    f"For a={report['a']}, b={report['b']}: P union Q covers D; "
    f"intersection count={report['overlap']}; "
    f"{report['sum_P']}+{report['sum_Q']}-{report['overlap']}={report['area']}. "
    "Finite verification; the notebook contains the general partition argument."
)
(OUTPUT / "floor-sum-workspace.json").write_text(workspace.to_json(), encoding="utf-8")
(OUTPUT / "floor-sum-cases.json").write_text(
    json.dumps([report, noncoprime_report], indent=2), encoding="utf-8")
for name, figure in [("partition", partition_plot), ("packing", packing_plot),
                     ("noncoprime", noncoprime_plot)]:
    if figure is not None:
        figure.write_html(OUTPUT / f"{name}.html", include_plotlyjs=True, auto_play=False)
print("Saved standalone interactive figures and the captured investigation to", OUTPUT)

The reusable pattern is **declare a universe → construct incidences → establish
coverage and overlap → count**. Changing the assumptions reveals exactly which
part of the argument changes. This is a concrete target for Kaleion's future
readable construction language and comparison statements.